#  Neural Networks — From Scratch to Training

This notebook builds a neural network step by step:
1. Single Neuron
2. Neural Network Architecture
3. Loss Functions
4. Gradient Descent & Backpropagation
5. Training Loop
6. Evaluation & Decision Boundary
7. CNN
8. Autoencoder


In [ ]:
# Install if needed (uncomment)
# !pip install -r requirements.txt
from typing import Literal

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, transforms
from pprint import pprint

# Reproducibility
np.random.seed(42)
torch.manual_seed(42);

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


---
## 1. The Building Blocks: Artificial Neuron

A neuron is a fundamental component that mimics a biological neuron.
It encodes the weighted sum of its inputs followed by a pass through an activation function, calculating how strongly to fire.

A single neuron computes:

$$z = \mathbf{w}^T \mathbf{x} + b = \sum_{i=1}^{n} w_i x_i + b$$

$$a = f(z)$$

Where:
- $\mathbf{x} \in \mathbb{R}^n$: input vector
- $\mathbf{w} \in \mathbb{R}^n$: weight vector
- $b \in \mathbb{R}$: bias scalar
- $f(\cdot)$: activation function
- $a$: neuron output (activation)


In [ ]:
def neuron(x, w, b, activation: Literal['relu', 'sigmoid', 'tanh', 'step', ''] = 'relu'):
    """Single artificial neuron: z = w^T x + b, a = f(z)"""
    z = np.dot(w, x) + b
    match activation:
        case 'relu':
            a = np.maximum(0, z)
        case 'sigmoid':
            a = 1 / (1 + np.exp(-z))
        case 'tanh':
            a = np.tanh(z)
        case 'step':  # Perceptron
            a = 1.0 if z >= 0 else 0.0
        case _:
            a = z
    return z, a

x = np.array([1.5, 2.0, -1.0])
w = np.array([0.4, -0.3, 0.8])
b = 0.1

z_val, a_val = neuron(x, w, b, activation='relu')
print(f'Input x     : {x}')
print(f'Weights w   : {w}')
print(f'Bias b      : {b}')
print(f'z = w·x + b : {z_val:.4f}')
print(f'a = ReLU(z) : {a_val:.4f}')


---
## 2. Neural Network in PyTorch
By arranging neurons in parallel we create one layer of neurons, and by chaining multiple layers in sequence we build a **Neural Network**.


The network transforms the input through three learned non-linear steps:

$$\mathbb{R}^{3} \xrightarrow{\text{Linear+ReLU}} \mathbb{R}^{4} \xrightarrow{\text{Linear+ReLU}} \mathbb{R}^{4} \xrightarrow{\text{Linear}} \mathbb{R}^{2}$$

Just like neurons in the brain form connections to pass signals forward,
each node in our network connects to every node in the next layer —
below we visualize these connections across all four layers.

![Neural Networn](./img/nn.png)

These layers of simple neurons are encoded in PyTorch using:
```py
nn.Linear(num_neurons_in_previous_layer, num_neurons_in_layer)
```

- Input layer: 3 features
- Hidden layer 1: 4 neurons + ReLU
- Hidden layer 2: 4 neurons + ReLU
- Output layer: 2 logits for binary classification


In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1  = nn.Linear(3, 4)   # input -> hidden 1
        self.fc2  = nn.Linear(4, 4)   # hidden 1 -> hidden 2
        self.out  = nn.Linear(4, 2)   # hidden 2 -> output
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.out(x)  # raw logits
        return x

model = NeuralNetwork().to(device)
print(model)

example = torch.tensor([3.0, 4.0, 5.0]).to(device)
logits  = model(example)
probs   = torch.softmax(logits, dim=0)
print(f'\nLogits       : {logits.detach()}')
print(f'Probabilities: {probs.detach()}')


---
## 3. Loss Functions — How Wrong Is the Model?

A loss function measures the difference between predictions and true labels — it's effectively a metric to measure correctness of prediction.

The goal of training is to **minimize this loss**.

### Cross-Entropy Loss (Classification)
$$\mathcal{L}_{CE} = -\sum_{c=1}^{C} y_c \log(\hat{y}_c)$$

PyTorch's `nn.CrossEntropyLoss` combines **Softmax + Log + NLL Loss** in one step.
Pass raw logits directly — no Softmax needed beforehand.


In [ ]:
criterion = nn.CrossEntropyLoss()

logits_batch = torch.tensor([
    [2.0, 0.5],   # confident → class 0
    [0.3, 1.8],   # confident → class 1
    [1.0, 1.0],   # uncertain
    [0.1, 0.9],   # leans → class 1
])
true_labels = torch.tensor([0, 1, 0, 1])

loss = criterion(logits_batch, true_labels)
probs_batch = torch.softmax(logits_batch, dim=1)

print(f'Logits:\n{logits_batch}')
print(f'True labels  : {true_labels.tolist()}')
print(f'Probabilities:\n{probs_batch.detach().numpy().round(3)}')
print(f'\nCross-Entropy Loss: {loss.item():.4f}')

# Visualization
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('white')

p = np.linspace(0.01, 0.99, 200)
ax[0].plot(p, -np.log(p),     color='#5C5EA6', lw=2, label='True label = 1')
ax[0].plot(p, -np.log(1 - p), color='#E05C5C', lw=2, label='True label = 0')
ax[0].set_xlabel('Predicted probability p')
ax[0].set_ylabel('Loss  −log(p)')
ax[0].set_title('Cross-Entropy Loss vs Confidence', fontweight='bold')
ax[0].legend(); ax[0].set_ylim(0, 5); ax[0].grid(alpha=0.3)

sample_losses = [criterion(logits_batch[i].unsqueeze(0),
                           true_labels[i].unsqueeze(0)).item() for i in range(4)]
bar_colors = ['#5C5EA6' if sl < 0.5 else '#E05C5C' for sl in sample_losses]
ax[1].bar([f'Sample {i+1}' for i in range(4)], sample_losses,
            color=bar_colors, edgecolor='white')
ax[1].axhline(loss.item(), color='gray', linestyle='--', label=f'Mean = {loss.item():.3f}')
ax[1].set_title('Per-Sample Loss', fontweight='bold')
ax[1].legend(); ax[1].grid(alpha=0.3, axis='y')

plt.tight_layout(); plt.show()


---
## 4. Gradient Descent & Backpropagation

Training adjusts weights to minimize the loss using **gradient descent**:

$$w \leftarrow w - \eta \cdot \nabla_w \mathcal{L}$$

Where $\eta$ is the **learning rate**. PyTorch computes gradients automatically
via `loss.backward()` (backpropagation), then `optimizer.step()` applies the update.

Every training iteration follows 4 steps:
1. **Forward pass** — compute predictions
2. **Loss** — measure error
3. **Backward pass** — compute gradients via backprop
4. **Update** — adjust weights with the optimizer


In [ ]:
# Dataset preparation
X, y = make_moons(n_samples=800, noise=0.20, random_state=42)

# Add 3rd feature so input_dim=3 matches our network
extra = np.random.normal(size=(X.shape[0], 1))
X = np.hstack([X, extra])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32).to(device)
y_test_t  = torch.tensor(y_test,  dtype=torch.long).to(device)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                          batch_size=32, shuffle=True)

print(f'Train samples: {len(X_train)} | Test samples: {len(X_test)}')


---
## 5. Training Loop


In [ ]:
model     = NeuralNetwork().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

def train_model(model, loader, criterion, optimizer, epochs=100):
    history = []
    model.train()
    for epoch in range(epochs):
        running_loss, correct, total = 0.0, 0, 0
        for xb, yb in loader:
            optimizer.zero_grad()          # 1. clear old gradients
            logits = model(xb)             # 2. forward pass
            loss   = criterion(logits, yb) # 3. compute loss
            loss.backward()                # 4. backprop
            optimizer.step()               # 5. update weights

            running_loss += loss.item() * xb.size(0)
            correct      += (logits.argmax(dim=1) == yb).sum().item()
            total        += yb.size(0)

        history.append((running_loss / total, correct / total))

    return history

history = train_model(model, train_loader, criterion, optimizer, epochs=100)
print(f'Final loss    : {history[-1][0]:.4f}')
print(f'Final accuracy: {history[-1][1]:.4f}')


In [ ]:
losses = [h[0] for h in history]
accs   = [h[1] for h in history]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('white')

ax[0].plot(losses, color='#E05C5C', lw=2)
ax[0].set_title('Training Loss', fontweight='bold')
ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss'); ax[0].grid(alpha=0.3)

ax[1].plot(accs, color='#5C5EA6', lw=2)
ax[1].set_title('Training Accuracy', fontweight='bold')
ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Accuracy')
ax[1].set_ylim(0, 1); ax[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()


---
## 6. Evaluation

### Train vs Eval Mode

In PyTorch, always switch modes explicitly:
- `model.train()` — enables training behaviour (Dropout, BatchNorm active)
- `model.eval()` — enables inference behaviour (Dropout off, BatchNorm frozen)
- `torch.no_grad()` — disables gradient tracking during inference (saves memory)

```python
# Training
model.train()
# ... training loop ...

# Evaluation
model.eval()
with torch.no_grad():
    preds = model(X_test)
```


In [ ]:
model.eval()
with torch.no_grad():
    test_logits = model(X_test_t)
    test_preds  = test_logits.argmax(dim=1)
    test_acc    = (test_preds == y_test_t).float().mean().item()

print(f'Test accuracy: {test_acc:.4f}')


In [ ]:
# Decision Boundary
model.eval()
h = 0.05
x_min, x_max = X_test[:, 0].min() - 1, X_test[:, 0].max() + 1
y_min, y_max = X_test[:, 1].min() - 1, X_test[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

grid   = np.c_[xx.ravel(), yy.ravel(), np.zeros(xx.ravel().shape[0])]
grid_t = torch.tensor(grid, dtype=torch.float32).to(device)

with torch.no_grad():
    Z = model(grid_t).argmax(dim=1).cpu().numpy().reshape(xx.shape)

plt.figure(figsize=(8, 5))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
plt.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='coolwarm', edgecolors='k', s=30)
plt.title('Decision Boundary on Test Set', fontweight='bold')
plt.tight_layout(); plt.show()


---
## 7. CNN — Convolutional Neural Network

A CNN is used for image-like inputs. It uses **convolution kernels** to detect
local patterns (edges, textures) instead of fully connected layers.

![CNN Representation](./img/cnn.png)

In [ ]:
# Create the transformation to be applied to the data
# Data is in PIL Image format
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,)) ])

# Get the training dataset from MNIST
train_data = datasets.MNIST(
    root='data',
    train=True,
    download=True,
    transform=transform
)

# Get the test dataset from MNIST
test_data = datasets.MNIST(
    root='data',
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(dataset=train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_data, batch_size=64, shuffle=True)

for test_images_batch, test_labels_batch in test_loader:
  print(test_images_batch.shape)
  print(test_labels_batch.shape)
  break

## Convolutional Network Processing Layers
***Convolutional Layers***: Defined in PyTorch using: <p>
`nn.Conv2d(num_channels, num_channels_out, kernel_size)`
But there are more important arguments like:
- Stride
- Padding

***MaxPool***: Save the biggest value in the moving window of size $K \times K$ <p>

***Dropout***: Zeroes the pixel with $X\%$ change, preventing neurons from becoming "lazy" by forcing each to learn to be "independent"

### Architecture:
- Convolutional Layers:<p>
`Conv2d → ReLU → Conv2d → ReLU → MaxPool (size k) → Dropout (25%)`
- Classifier (Fully Connected) Layers: <p>
` Flatten → Linear → Relu → Output Layer`

To calculate the number of inputs to the initial classification layer we use the following formula:

$$O_{width} = \lfloor \frac{W - K_w + 2P_w}{S_w} \rfloor + 1$$
$$O_{height} = \lfloor \frac{H - K_h + 2P_h}{S_h} \rfloor + 1$$



In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(9216, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

print(SimpleCNN(num_classes=10))

In [ ]:
model     = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5

model.to(device)
history = []
num_examples = len(train_data)
num_batches = len(train_loader)
for epoch in tqdm(range(epochs), unit="epochs"):
    model.train()
    epoch_loss = 0
    correct = 0
    for images_batch, labels_batch in train_loader:
        # Send batch to GPU
        images_batch, labels_batch = images_batch.to(device), labels_batch.to(device)

        # 1. clear old gradients
        optimizer.zero_grad()
        # 2. forward pass
        pred = model(images_batch)
        # 3. compute loss
        loss   = criterion(pred, labels_batch)
        epoch_loss += loss.item()
        correct += (pred.argmax(dim=1) == labels_batch).sum().item()

        # 4. backprop
        loss.backward()
        # 5. update weights
        optimizer.step()

    # ---- VALIDATE ----
    model.eval()
    val_loss = 0.0
    val_correct = 0

    with torch.no_grad():
        for images_batch, labels_batch in test_loader:
            images_batch, labels_batch = images_batch.to(device), labels_batch.to(device)

            pred = model(images_batch)
            loss = criterion(pred, labels_batch)

            val_loss += loss.item()
            val_correct += (pred.argmax(dim=1) == labels_batch).sum().item()

    val_loss /= len(test_loader)
    val_acc  = val_correct / len(test_data)

    train_loss = epoch_loss / num_batches
    train_acc  = correct / num_examples
    history.append((train_loss, train_acc, val_loss, val_acc))

print("\nEpoch's Average Loss and Accuracy")
for i, (loss, acc, val_loss, val_acc) in enumerate(history):
    print(f"Epoch {i+1:>2d} - Training loss={loss:.3f} accuracy={acc:.3f} | Test loss={val_loss:.3f} accuracy={val_acc:.3f}")

In [ ]:
# Plot training/test loss and training/test accuracy from CNN history
train_losses = [h[0] for h in history]
train_accs   = [h[1] for h in history]
test_losses  = [h[2] for h in history]
test_accs    = [h[3] for h in history]

epochs_range = range(1, len(history) + 1)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor("white")

# Loss curves
ax[0].plot(epochs_range, train_losses, marker="o", label="Train Loss")
ax[0].plot(epochs_range, test_losses, marker="o", label="Test Loss")
ax[0].set_title("Loss vs Epoch")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].grid(alpha=0.3)
ax[0].legend()

# Accuracy curves
ax[1].plot(epochs_range, train_accs, marker="o", label="Train Accuracy")
ax[1].plot(epochs_range, test_accs, marker="o", label="Test Accuracy")
ax[1].set_title("Accuracy vs Epoch")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Accuracy")
ax[1].set_ylim(0, 1)
ax[1].grid(alpha=0.3)
ax[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
model.to('cpu')
model.eval()
n=0
figure = test_images_batch[n:n+1]
expected = test_labels_batch[n:n+1]
with torch.no_grad():
    output = model(figure)
    predicted_class = output.argmax()

# Show image .permute changes the coordinates from (channel, x, y) to (x, y, channel)
plt.imshow(figure[0].permute(1, 2, 0).numpy())
print("Expected class:", expected.item())
print("Predicted class:", predicted_class.item())

---
## 8. Encoder-Decoder / Autoencoder

An autoencoder is an unsupervised learning algorithm that learns how to **compress** inputs into a lower dimensional vector, a latent vector.
This section of the neural network is called the "Bottleneck"

Afterwards, it attempts to **reconstruct** the original input based only on the latent vector.

Training objective:
- Create a compressed embedding representation of the input.
- Minimize reconstruction error using **MSELoss**.

$$\mathcal{L}_{MSE} = \frac{1}{n}\sum_{i=1}^{n}(x_i - \hat{x}_i)^2$$


This architecture is useful for
- Dimensionality Reduction
- Denoising data: Models are forced to perserve only the most relevant aspects of the data
- Anomaly Detection: Model learns how to encode and decode healthy/normal data, but fails at encoding and decoding anomaly data

In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3),
            nn.ReLU()
        )
        
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(32, 16, 3),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 1, 3),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [ ]:
model = ConvAutoencoder().to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10
history = []

for epoch in tqdm(range(epochs), desc="Epochs"):
    model.train()
    epoch_loss = 0.0
    
    for images_batch, _ in train_loader:
        images_batch = images_batch.to(device)

        # 1. Clear out gradient        
        optimizer.zero_grad()
        
        # 2. Forward pass
        reconstructed = model(images_batch)
        
        # 3. Compute loss against original input
        loss = criterion(reconstructed, images_batch)
        epoch_loss += loss.item()
        
        # 4. Backprop and update weights
        loss.backward()
        optimizer.step()
        
    model.eval()
    val_loss = 0.0
    
    with torch.no_grad():
        for images_batch, _ in test_loader:
            images_batch = images_batch.to(device)
            
            reconstructed = model(images_batch)
            loss = criterion(reconstructed, images_batch)
            
            val_loss += loss.item()
            
    train_loss_avg = epoch_loss / len(train_loader)
    val_loss_avg = val_loss / len(test_loader)
    history.append((train_loss_avg, val_loss_avg))

print("\nEpoch | Train Loss | Test Loss")
print("---------------------------------")
for i, (t_loss, v_loss) in enumerate(history):
    print(f"{i+1:5d} | {t_loss:.4f}     | {v_loss:.4f}")

In [ ]:
# Plot training/test loss and training/test accuracy from CNN history
train_losses = [h[0] for h in history]
test_losses  = [h[1] for h in history]

epochs_range = range(1, len(history) + 1)

fig, ax = plt.subplots(1, 1, figsize=(12, 4))
fig.patch.set_facecolor("white")

# Loss curves
ax.plot(epochs_range, train_losses, marker="o", label="Train Loss")
ax.plot(epochs_range, test_losses, marker="o", label="Test Loss")
ax.set_title("Loss vs Epoch")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.grid(alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
model.to('cpu')
model.eval()
n=0
figure = test_images_batch[n:n+1]
expected = test_labels_batch[n:n+1]
with torch.no_grad():
    reconstruction = model(figure)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor("white")
# Show image .permute changes the coordinates from (channel, x, y) to (x, y, channel)
ax[0].imshow(figure[0].permute(1, 2, 0).numpy())
ax[0].set_title(f"Original Image (Label: {expected.item()})")
ax[1].imshow(reconstruction[0].permute(1, 2, 0).numpy())
ax[1].set_title(f"Reconstruction (Label: {expected.item()})")
plt.show()

In [ ]:
class AnomalyDetectorAE(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder: Compresses the 3 input features into a smaller latent space
        self.encoder = nn.Sequential(
            nn.Linear(3, 2),
            nn.ReLU(),
            nn.Linear(2, 1),
            nn.ReLU()
        )
        # Decoder: Attempts to reconstruct the original 3 features
        self.decoder = nn.Sequential(
            nn.Linear(1, 2),
            nn.ReLU(),
            nn.Linear(2, 3)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, x):
        return self.decoder(x)
    
model = AnomalyDetectorAE()
print(model)

In [ ]:
criterion = nn.MSELoss()

normal_events = torch.tensor([
    [1.0, 20.0, 300.0],
    [0.0, 18.0, 280.0],
    [1.0, 22.0, 310.0]
], dtype=torch.float32)

mu  = normal_events.mean()
sigma = normal_events.std()
normal_events = (normal_events - mu) / sigma

epochs = 150
for epoch in tqdm(range(epochs), desc="Elapsed epochs", unit="epoch"):
    optimizer.zero_grad()
    reconstructed = model(normal_events)
    loss = criterion(reconstructed, normal_events)
    loss.backward()
    optimizer.step()

In [ ]:
ind_mse_loss = nn.MSELoss(reduction='none')

suspicious_event = torch.tensor([
    [7.0, 95.0, 1200.0]
], dtype=torch.float32)


model.eval()
with torch.no_grad():
    # Calculate reconstruction error for normal events
    recon_norm_events = model(normal_events)
    normal_events_errors = ind_mse_loss(normal_events, recon_norm_events).mean(dim=1)

    # Calculate reconstruction error for suspicious event
    recon_suspicious_event = model(suspicious_event)
    error_suspicious_event = ind_mse_loss(suspicious_event, recon_suspicious_event)

# Calculate the average reconstruction error for normal events
average_normal_error = normal_events_errors.mean().item()
print("Final Anomaly Scores:")
for i, score in enumerate(normal_events_errors):
    error = score.item()
    delta_error = error - average_normal_error
    print(f"Normal Event {i}: {delta_error:.4f}")

# Calculate threshold for anomaly detection
threshold = mu + (2 * sigma)
if threshold < error_suspicious_event.mean().item():
    print("Anomaly detected!")
    print(f"Suspicious Event: {error_suspicious_event.mean().item() - average_normal_error:.4f}")